# 文本分类实例

## Step1 导入相关包


In [1]:
from transformers import AutoTokenizer,AutoModelForSequenceClassification

c:\Users\Aaron\AppData\Local\Programs\miniconda3\envs\Bert\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step2 加载数据

In [2]:
import pandas as pd

data = pd.read_csv("./data/ChnSentiCorp_htl_all.csv")
data.head()

,label,review
0,1,"距离川沙公路较近,但是公交指示不对,如果是""蔡陆线""的话,会非常麻烦.建议用别的路线.房间较..."
1,1,商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!
2,1,早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很好。
3,1,宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很小，确实挺小...
4,1,"CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风"


In [3]:
data = data.dropna()
data

,label,review
0,1,"距离川沙公路较近,但是公交指示不对,如果是""蔡陆线""的话,会非常麻烦.建议用别的路线.房间较..."
1,1,商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!
2,1,早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很好。
3,1,宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很小，确实挺小...
4,1,"CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风"
...,...,...
7761,0,尼斯酒店的几大特点：噪音大、环境差、配置低、服务效率低。如：1、隔壁歌厅的声音闹至午夜3点许...
7762,0,盐城来了很多次，第一次住盐阜宾馆，我的确很失望整个墙壁黑咕隆咚的，好像被烟熏过一样家具非常的...
7763,0,看照片觉得还挺不错的，又是4星级的，但入住以后除了后悔没有别的，房间挺大但空空的，早餐是有但...
7764,0,我们去盐城的时候那里的最低气温只有4度，晚上冷得要死，居然还不开空调，投诉到酒店客房部，得到...


## Step3 创建Dataset

In [4]:
from torch.utils.data import Dataset

class MyDataset(Dataset):

    def __init__(self) -> None:
        super().__init__()
        self.data = pd.read_csv("./data/ChnSentiCorp_htl_all.csv")
        self.data = self.data.dropna()
    
    def __getitem__(self, index):
        return self.data.iloc[index]["review"],self.data.iloc[index]["label"]
    
    def __len__(self):
        return len(self.data)

In [5]:
dataset = MyDataset()
for i in range(5):
    print(dataset[i])

('距离川沙公路较近,但是公交指示不对,如果是"蔡陆线"的话,会非常麻烦.建议用别的路线.房间较为简单.', np.int64(1))
('商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!', np.int64(1))
('早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很好。', np.int64(1))
('宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很小，确实挺小，但加上低价位因素，还是无超所值的；环境不错，就在小胡同内，安静整洁，暖气好足-_-||。。。呵还有一大优势就是从宾馆出发，步行不到十分钟就可以到梅兰芳故居等等，京味小胡同，北海距离好近呢。总之，不错。推荐给节约消费的自助游朋友~比较划算，附近特色小吃很多~', np.int64(1))
('CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风', np.int64(1))


## Step4 划分数据集

In [6]:
import torch
from torch.utils.data import random_split

# 三划分
train_len = int(0.8 * len(dataset))
valid_len = int(0.1 * len(dataset))
test_len = len(dataset) - train_len - valid_len

trainset, validset, testset = random_split(
    dataset,
    [train_len, valid_len, test_len],
    generator=torch.Generator().manual_seed(42)
)
len(trainset),len(validset),len(testset)

(6212, 776, 777)

In [7]:
for i in range(10):
    print(trainset[i])

('房间很好，符合五星级的标准。酒店离沟口及边边街都比较近，是去九寨沟旅游的最佳选择。补充点评2008年7月21日：地震对该酒店没有任何影响。宾馆反馈2008年7月24日：谢谢您的点评,欢迎下次下榻我酒店!我们将为您提供更周到的服务及优质的产品.', np.int64(1))
('性价比还不错。。。算是当地最好的酒店了如果是第一次去的话，可能会发现不太好找', np.int64(1))
('打开评价页面就很不一样，很少有酒店能这么认真的作评价回复，这一点就很不错了。我们是A9上海至湖州通车那天来湖州逛逛的，定了太湖豪华房，应该算是不错，建筑比较吸引人，不过那天风大雨大就一直在房间里，可能观念的问题吧，就五星标准来说，房间足够大，浴室也做的很漂亮，但却少许多东西，感觉空落落的，床上用品也只有简单的两个枕头、一条薄被，看上去就好冷啊~应该会比较适合夏季度假吧：）由于临时有事，没有过夜就提前退房了，前台mm还很遗憾的说，下次一定要来住哦，听了蛮窝心的，我想应该会在暖和些的日子再来的：）也希望再次入住前酒店能做更好的改进，哈哈补充点评2008年1月17日：关键没有国际卫视，这一点作为五星标准比较令人不能接受：S', np.int64(1))
('5月14日入住了这家酒店，服务确实很好，前台会亲自带领客人到房间，并介绍房间设施，得知我自带电脑，立刻把网线从房间内电脑上拆下，放在桌旁待用。晚上六点后会送果盘和小礼物，那天是香蕉和小西红柿，两个毛绒公仔。房间相较其他济南同价位的酒店显得新，很干净，位置也很好！推荐！', np.int64(1))
('位置是非常好，就在省政府对面，离步行街也很近，但房间内的条件太一般，达不到四星。我住的是398元的，还没包早餐，最多也就值200元，不过酒店能到机场接客确实很不错。如果住一晚也值了，如果住三晚以上，还是选择别的实惠一点的酒店吧。现在我才发现，通过携程网订房价格很没优势的，都是提高了价格后再折扣，折后价格比不上其他的普通酒店，以后还是要自己直接跟别的酒店联系，省了中间环节，价格会低很多，并且条件还会更好。', np.int64(1))
('1、我原本是通过丽江某旅行社订房在香格里拉大道某酒店，到达酒店后，对酒店不甚满意。出租车司机向我推荐了古城里的“丽王酒店”、“王府饭店”、“和玺酒店”。对比三家酒店，和玺酒店更具有古朴特色，一个个院落

## Step5 创建Dataloader

In [8]:
import torch

tokenizer = AutoTokenizer.from_pretrained("hfl/rbt3")

def collate_func(batch):
    texts,labels = [],[]
    for item in batch:
        texts.append(item[0])
        labels.append(item[1])
    inputs = tokenizer(texts,max_length=128,padding="max_length",truncation=True,return_tensors="pt", dtype=torch.long)
    inputs["labels"] = torch.tensor(labels)
    return inputs

In [9]:
from torch.utils.data import DataLoader

trainloader = DataLoader(trainset,batch_size=32,shuffle=True,collate_fn=collate_func)
validloader = DataLoader(validset,batch_size=64,shuffle=False,collate_fn=collate_func)
testloader = DataLoader(testset,batch_size=64,shuffle=False,collate_fn=collate_func)

In [10]:
next(enumerate(trainloader))

(0,
 {'input_ids': tensor([[ 101, 6821, 3613,  ...,    0,    0,    0],
         [ 101, 1922, 2345,  ...,    0,    0,    0],
         [ 101,  679, 3221,  ...,    0,    0,    0],
         ...,
         [ 101, 1057,  857,  ...,    0,    0,    0],
         [ 101,  122,  510,  ..., 6421, 8013,  102],
         [ 101, 6983, 2421,  ...,    0,    0,    0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         ...,
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         ...,
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 0, 0, 0]]), 'labels': tensor([1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 0, 1, 1, 1, 1, 1, 1])})

## Step6 创建模型及优化器

In [11]:
import torch.nn as nn
from transformers import AutoModel


class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = AutoModel.from_pretrained("hfl/rbt3")

        # 冻结前2层
        for param in self.bert.encoder.layer[:2].parameters():
            param.requires_grad = False
            
        # self.classifier = nn.Linear(768, 2)    
        # 改进分类头（MLP）
        self.classifier = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 2)
        )

    def forward(self, input_ids, attention_mask, token_type_ids=None, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)

        # CLS向量
        cls = outputs.last_hidden_state[:, 0]

        logits = self.classifier(cls)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)

        return {"loss": loss, "logits": logits}

In [12]:
from torch.optim import Adam


model = MyModel().cuda()
optimizer = Adam(model.parameters(), lr=2e-5)

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 49716.97it/s]
BertModel LOAD REPORT from: hfl/rbt3
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Step7 训练与验证

In [13]:
import numpy as np


def evaluate():
    model.eval()
    acc_num = 0
    preds, labels = [], []
    with torch.inference_mode():
        for batch in validloader:
            if torch.cuda.is_available():
                batch = {k: v.cuda() for k,v in batch.items()}
            output = model(**batch)
            pred = torch.argmax(output['logits'],dim=-1)
            preds.extend(pred.cpu().tolist())
            labels.extend(batch["labels"].cpu().tolist())
            acc_num += (pred.long() == batch["labels"].long()).float().sum()
    
    print(np.mean(preds), np.mean(labels))
    # print(confusion_matrix(labels, preds))
    return acc_num / len(validset)

def train(epoch=3,log_step=100):
    global_step = 0
    for ep in range(epoch):
        model.train()
        for batch in trainloader:
            if torch.cuda.is_available():
                batch = {k: v.cuda() for k,v in batch.items()}
            optimizer.zero_grad()
            output = model(**batch)
            output['loss'].backward()
            optimizer.step()
            if global_step % log_step == 0:
                print(f"ep: {ep},global_step: {global_step},loss: {output['loss'].item()}")
            global_step += 1
        acc = evaluate()
        print(f"ep : {ep}, acc : {acc}")

In [14]:
# 不训练，直接evaluate
acc = evaluate()
print("微调前准确率:", acc)

Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "c:\Users\Aaron\AppData\Local\Programs\miniconda3\envs\Bert\Lib\site-packages\huggingface_hub\utils\_http.py", line 761, in hf_raise_for_status
    response.raise_for_status()
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "c:\Users\Aaron\AppData\Local\Programs\miniconda3\envs\Bert\Lib\site-packages\httpx\_models.py", line 829, in raise_for_status
    raise HTTPStatusError(message, request=request, response=self)
httpx.HTTPStatusError: Client error '403 Forbidden' for url 'https://huggingface.co/api/models/hfl/rbt3/discussions?p=0'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "c:\Users\Aaron\AppData\Local\Programs\miniconda3\envs\Bert\Lib\threading.py", line 1082, in _bootstrap_inner
    self._context.run(self.run)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^
  File "

0.03608247422680412 0.7164948453608248
微调前准确率: tensor(0.3093, device='cuda:0')


In [15]:
train()

ep: 0,global_step: 0,loss: 0.7144602537155151
ep: 0,global_step: 100,loss: 0.45481786131858826
0.7100515463917526 0.7164948453608248
ep : 0, acc : 0.844072163105011
ep: 1,global_step: 200,loss: 0.36778324842453003
ep: 1,global_step: 300,loss: 0.21757790446281433
0.7474226804123711 0.7164948453608248
ep : 1, acc : 0.8685566782951355
ep: 2,global_step: 400,loss: 0.41955631971359253
ep: 2,global_step: 500,loss: 0.1645175665616989
0.6932989690721649 0.7164948453608248
ep : 2, acc : 0.8814432621002197


In [16]:
# 训练后，再次evaluate
acc = evaluate()
print("微调后准确率:", acc)

0.6932989690721649 0.7164948453608248
微调后准确率: tensor(0.8814, device='cuda:0')


In [17]:
import numpy as np


def final_test():
    model.eval()
    acc_num = 0
    preds, labels = [], []
    with torch.inference_mode():
        for batch in testloader:
            if torch.cuda.is_available():
                batch = {k: v.cuda() for k,v in batch.items()}
            output = model(**batch)
            pred = torch.argmax(output['logits'],dim=-1)
            preds.extend(pred.cpu().tolist())
            labels.extend(batch["labels"].cpu().tolist())
            acc_num += (pred.long() == batch["labels"].long()).float().sum()
    
    print(np.mean(preds), np.mean(labels))
    # print(confusion_matrix(labels, preds))
    return acc_num / len(testset)

final_acc = final_test()
print("最终测试集准确率:", final_acc)

0.6872586872586872 0.6885456885456885
最终测试集准确率: tensor(0.8958, device='cuda:0')


In [18]:
import gc

gc.collect()
torch.cuda.empty_cache()

In [ ]:
print(model)

In [ ]:
model.save_pretrained("./rbt3_sentiment_model")

## Step9 模型预测

In [ ]:
sen = "我觉得这家酒店不错，饭很好吃"
id2_label = {0 : "差评!",1 :"好评!"}

model.eval()
with torch.inference_mode():
    inputs = tokenizer(sen,return_tensors="pt")
    inputs = {k : v.cuda() for k,v in inputs.items()}
    logits = model(**inputs).logits
    pred = torch.argmax(logits,dim=-1)
    print(f"输入: {sen} \n 模型预测结果: {id2_label.get(pred.item())}")

In [ ]:
import gradio as gr

id2_label = {0: "差评!", 1: "好评!"}

def predict_sentiment(text):
    model.eval()
    with torch.inference_mode():
        inputs = tokenizer(text, return_tensors="pt", max_length=128, padding="max_length", truncation=True)
        inputs = {k: v.cuda() for k, v in inputs.items()}
        logits = model(**inputs).logits
        pred = torch.argmax(logits, dim=-1)
        label = id2_label.get(pred.item())
        confidence = torch.softmax(logits, dim=-1)[0][pred.item()].item()
        return f"{label} (置信度: {confidence:.2f})"

iface = gr.Interface(
    fn=predict_sentiment,
    inputs=gr.Textbox(lines=3, placeholder="请输入您要分析的酒店评论..."),
    outputs="text",
    title="酒店评论情感分析",
    description="输入一段酒店评论，模型将预测其情感倾向（好评或差评）。"
)

iface.launch()

In [ ]:
from transformers import pipeline

model.config.id2label = id2_label
pipe = pipeline("text-classification",model=model,tokenizer=tokenizer,device=0)

In [ ]:
pipe(sen)